# 嘉兴大学 2026 级新生人数 · 旭日图

- 数据：`data/新生人数.csv`（学院 / 专业 / 人数，合计 4638）
- 交互：点击学院扇区 → 下钻到该学院的专业；悬停看 `人数 / 占本层 / 占总体`
- 环境：JupyterLab + pyecharts 2.x


## 1. 读数据

CSV 两列层级：学院 → 专业。按学院人数降序排列，图上从 12 点方向顺时针由大到小铺开。

In [1]:
import csv
from pathlib import Path

from pyecharts import options as opts
from pyecharts.charts import Sunburst
from pyecharts.commons.utils import JsCode

CSV_PATH = Path("data/新生人数.csv")

groups = {}
with open(CSV_PATH, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        groups.setdefault(row["学院"], []).append((row["专业"], int(row["人数"])))

# 学院按人数降序，专业内部也按人数降序
data = [(c, sorted(v, key=lambda x: -x[1])) for c, v in groups.items()]
data.sort(key=lambda x: -sum(n for _, n in x[1]))

TOTAL = sum(n for _, majors in data for _, n in majors)
print(f"{len(data)} 个学院，{sum(len(v) for _, v in data)} 个专业，合计 {TOTAL} 人")

15 个学院，61 个专业，合计 4638 人


## 2. 拼成旭日图的数据树

根节点 = 全校（圆心那一块），第一层 = 学院，第二层 = 专业。
每个学院一个色系，专业用同色系浅色，视觉上自然成块。

In [2]:
PALETTE = [
    "#5470c6", "#91cc75", "#fac858", "#ee6666", "#73c0de",
    "#3ba272", "#fc8452", "#9a60b4", "#ea7ccc", "#b6a2de",
    "#5ab1ef", "#ffb980", "#d87a80", "#8d98b3", "#e5cf0d",
]


def mix(color, other, t):
    """把 color 往 other 混 t（0~1），生成同色系浅色"""
    a = [int(color[i:i + 2], 16) for i in (1, 3, 5)]
    b = [int(other[i:i + 2], 16) for i in (1, 3, 5)]
    return "#" + "".join(f"{round(x + (y - x) * t):02x}" for x, y in zip(a, b))


children = []
for i, (college, majors) in enumerate(data):
    base = PALETTE[i % len(PALETTE)]
    children.append({
        "name": college,
        "value": sum(n for _, n in majors),
        "itemStyle": {"color": base},
        "children": [
            {
                "name": major,
                "value": num,
                "itemStyle": {"color": mix(base, "#ffffff", 0.62 + 0.05 * (j % 3))},
            }
            for j, (major, num) in enumerate(majors)
        ],
    })

tree = [{
    "name": "2026级总览",
    "value": TOTAL,
    "itemStyle": {"color": "#ffffff"},   # 圆心留白
    "label": {"rotate": 0},              # 圆心文字水平
    "children": children,
}]
len(children), children[0]["name"], children[0]["value"]

(15, '商学院', 549)

## 3. 画图

- `radius=["0%", "90%"]`：三层等分，圆心 0~30% 是总览，30~60% 学院，60~90% 专业
- 点击学院 → 以它为根重画两层树（参考图的下钻效果）；点圆心回总览
- 不用 ECharts 内置的 `nodeClick="rootToNode"`，因为它会把「全校」那层也留在圆心，下钻后变三层
- tooltip 用 `value / 父节点人数` 与 `value / 4638` 直接算比值，和参考图字段一致

In [3]:
LABEL_JS = """function (p) {
    var share = p.value / %d;
    if (share < 0.004) { return ''; }
    return p.name + '\\n' + p.value;
}""" % TOTAL

TOOLTIP_JS = """function (p) {
    var path = p.treePathInfo || [];
    var parent = path.length > 1 ? path[path.length - 2].value : p.value;
    return p.name
        + '<br/>人数: ' + p.value
        + '<br/>占本层: ' + (p.value / parent)
        + '<br/>占总体: ' + (p.value / %d);
}""" % TOTAL

from sunburst_repro import drill_js          # 点击下钻的辅助脚本（与 .py 同一份）

chart = Sunburst(
    init_opts=opts.InitOpts(width="1000px", height="900px", bg_color="#ffffff")
)
chart.add(
    series_name="",
    data_pair=tree,
    radius=["0%", "90%"],
    center=["50%", "50%"],
    sort_="desc",
    is_clockwise=True,
    start_angle=90,
    label_opts=opts.LabelOpts(
        is_show=True,
        rotate="radial",
        color="#333333",
        font_size=10.5,
        formatter=JsCode(LABEL_JS),
    ),
    itemstyle_opts=opts.ItemStyleOpts(border_color="#ffffff", border_width=1.5),
)
chart.set_global_opts(
    title_opts=opts.TitleOpts(is_show=False),
    legend_opts=opts.LegendOpts(is_show=False),
    tooltip_opts=opts.TooltipOpts(
        trigger="item",
        formatter=JsCode(TOOLTIP_JS),
    ),
)
# 长标签截断，避免顶到扇区外面
chart.options["series"][0]["label"].update(width=110, overflow="truncate")
chart.options["animation"] = False

# 点学院下钻（以该学院为根重画两层树）、点圆心回总览
chart.add_js_funcs(drill_js(chart.chart_id))

chart.render_notebook()

> 提示：下钻后想回到总览，刷新页面或重新运行本单元格即可。